In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs\\ablations_hpo"
FILE_NAME = f"ablation_A4_1s3w_hpo_v1"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: ablation_A4_1s3w_hpo_v1
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\ablations_hpo\ablation_A4_1s3w_hpo_v1.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'ablation_A4_1s3w_hpo_v1'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 100 trials.
Best value (Accuracy): 0.9294
Best Hyperparameters:
  outer_lr: 5.885329429123307e-05
  wd: 0.00045391358721408743
  maml_inner_steps: 15
  maml_alpha_init: 0.023655892884672083
  maml_alpha_init_eval: 0.04628880659796761
  maml_use_lslr: True
  use_lslr_at_eval: True
  use_maml_msl: hybrid
  maml_msl_num_epochs: 24
  episodes_per_epoch_train: 100


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL

#params_to_plot_ARCH = [
#    "cnn_base_filters", "lstm_hidden", "groupnorm_num_groups", "use_GlobalAvgPooling"
#]

params_to_plot_OPT = [
    "outer_lr", "wd", #"label_smooth", "ft_lr"
]

params_to_plot_MAML = [
    "maml_inner_steps", "maml_alpha_init_eval", "maml_use_lslr", "use_lslr_at_eval", "use_maml_msl", "episodes_per_epoch_train"
]

In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_OPT)
fig_slice.show()

In [11]:
fig_slice = plot_slice(study, params=params_to_plot_MAML)
fig_slice.show()

In [12]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
82,82,0.929375,None,2026-04-20 15:24:44.766690,0 days 00:14:41.809615
90,90,0.928646,None,2026-04-20 15:43:45.533564,0 days 00:11:34.699474
76,76,0.926979,None,2026-04-20 15:08:03.102765,0 days 00:14:59.714984
86,86,0.925417,None,2026-04-20 15:39:25.005109,0 days 00:15:05.269937
84,84,0.925104,None,2026-04-20 15:28:01.724920,0 days 00:15:01.732872
35,35,0.924479,None,2026-04-20 13:24:47.687979,0 days 00:14:43.671807
83,83,0.924375,None,2026-04-20 15:25:49.376497,0 days 00:14:02.325924
74,74,0.923438,None,2026-04-20 14:54:41.680380,0 days 00:14:39.324060
97,97,0.923021,None,2026-04-20 15:53:37.783471,0 days 00:29:47.934432
85,85,0.922708,None,2026-04-20 15:39:24.784775,0 days 00:13:25.059515


In [13]:
from optuna.trial import TrialState

# 1. Filter for only successfully completed trials
completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]

# 2. Sort the filtered list (using reverse=True if you want the highest values)
top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:10]

# 3. Print the params
for t in top_trials:
    print(t.params)

{'outer_lr': 5.885329429123307e-05, 'wd': 0.00045391358721408743, 'maml_inner_steps': 15, 'maml_alpha_init': 0.023655892884672083, 'maml_alpha_init_eval': 0.04628880659796761, 'maml_use_lslr': True, 'use_lslr_at_eval': True, 'use_maml_msl': 'hybrid', 'maml_msl_num_epochs': 24, 'episodes_per_epoch_train': 100}
{'outer_lr': 0.00011092306756347356, 'wd': 5.5908242475526956e-05, 'maml_inner_steps': 15, 'maml_alpha_init': 0.02286706275574153, 'maml_alpha_init_eval': 0.036881587994404635, 'maml_use_lslr': False, 'use_lslr_at_eval': True, 'use_maml_msl': 'hybrid', 'maml_msl_num_epochs': 26, 'episodes_per_epoch_train': 100}
{'outer_lr': 7.259492459259646e-05, 'wd': 0.0002757147623134879, 'maml_inner_steps': 15, 'maml_alpha_init': 0.01974520201579346, 'maml_alpha_init_eval': 0.03128188387867633, 'maml_use_lslr': False, 'use_lslr_at_eval': True, 'use_maml_msl': 'hybrid', 'maml_msl_num_epochs': 24, 'episodes_per_epoch_train': 100}
{'outer_lr': 6.992345661286494e-05, 'wd': 0.00013953333025680612, 